# 1.2 — Forecast Accuracy

- **Propósito:** Comparar o forecast de três meses com a demanda realizada.
- **Entrada:** `parts_hdbk_sandbox.pr_demand.demand_analytical_base`, `parts_hdbk_sandbox.pr_forecast.refined_forecast_enriched`
- **Saída:** Base comparativa entre forecast e demanda realizada
- **Chave:** `segment` + `market` + `main_material` + `forecast_month` · **Carga:** Sob demanda

In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "parts_hdbk_sandbox"
DEMAND_TABLE = f"{CATALOG}.pr_demand.demand_analytical_base"
FORECAST_TABLE = f"{CATALOG}.pr_forecast.refined_forecast_enriched"
FORECAST_LAG = 3
JOIN_KEYS = ["segment", "market", "main_material", "forecast_month"]

print(f"Demanda  : {DEMAND_TABLE}")
print(f"Forecast : {FORECAST_TABLE} (lag={FORECAST_LAG})")

In [0]:
df_demand = spark.table(DEMAND_TABLE)

In [0]:
df_forecast = spark.table(FORECAST_TABLE)

In [0]:
df_forecast_agg = (
    df_forecast
    .filter(F.col("lag") == FORECAST_LAG)
    .groupBy(*JOIN_KEYS)
    .agg(F.sum("forecast_qty").alias("forecast_qty"))
)

In [0]:
# Códigos 01 e 02 representam mercados Domestic e Export.
df_demand_agg = (
    df_demand
    .select(
        F.col("organizacao_vendas").alias("segment"),
        F.when(F.col("canal_distribuicao") == "01", "Domestic")
        .when(F.col("canal_distribuicao") == "02", "Export")
        .otherwise(F.col("canal_distribuicao"))
        .alias("market"),
        F.col("familia_produto").alias("main_material"),
        F.trunc("data_ordem", "month").alias("forecast_month"),
        "quantidade",
    )
    .groupBy(*JOIN_KEYS)
    .agg(F.sum("quantidade").alias("actual_qty"))
)

In [0]:
df_accuracy = (
    df_forecast_agg
    .join(df_demand_agg, JOIN_KEYS, "inner")
    .select(*JOIN_KEYS, "forecast_qty", "actual_qty")
)

assert df_accuracy.columns == [*JOIN_KEYS, "forecast_qty", "actual_qty"]

matched = df_accuracy.count()
print(f"Registros com match (inner join): {matched:,}")
display(df_accuracy.orderBy("segment", "main_material", "forecast_month").limit(10))